In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
# !wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
!cp /content/drive/MyDrive/MMDS-data/spark-3.1.1-bin-hadoop3.2.tgz .
!tar xf spark-3.1.1-bin-hadoop3.2.tgz
!pip install -q findspark

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!cp /content/drive/MyDrive/MMDS-data/spark-3.1.1-bin-hadoop3.2.gz .
!tar xf spark-3.1.1-bin-hadoop3.2.gz
!pip install -q findspark

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"
import findspark
findspark.init()

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
        .appName("task1-endterm") \
        .config("spark.executor.memory", "4g") \
        .config("spark.driver.memory", "4g") \
        .getOrCreate()

sc = spark.sparkContext
ratings2k = "/content/drive/MyDrive/MMDS-data/ratings2k.csv"

**Task 1: Dimensionality Reduction – SVD**

In [ ]:
from pyspark.sql import functions as F

def load_data(input_path=ratings2k):
    df = spark.read.csv(input_path, header=True, inferSchema=True)
    user_item_matrix = df.groupBy("user").pivot("item").agg(F.first("rating")).na.fill(0)
    return user_item_matrix

user_item_matrix = load_data()
user_item_matrix.show()

+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+--

In [ ]:
from pyspark.mllib.linalg import SparseVector
from pyspark.mllib.linalg.distributed import RowMatrix
from pyspark.ml.feature import VectorAssembler

def computeSVD(user_item_matrix, num_concepts=32, feature_columns=None):
    # Determine feature columns
    if feature_columns is None:
        feature_columns = [col for col in user_item_matrix.columns if col != "user"]

    # Assemble feature vectors
    assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
    df_with_features = assembler.transform(user_item_matrix).select("features")

    # Convert DenseVector to SparseVector in RDD
    def to_sparse_vector(dense_vector):
        return SparseVector(len(dense_vector), [(i, v) for i, v in enumerate(dense_vector) if v != 0])

    sparse_vectors_rdd = df_with_features.rdd.map(lambda row: to_sparse_vector(row.features))
    sparse_vectors_sample = sparse_vectors_rdd.take(10)

    print("Sample sparse vectors:")
    for i, vec in enumerate(sparse_vectors_sample):
        print(f"Sparse Vector {i + 1}: {vec}")

    # Create RowMatrix
    row_matrix = RowMatrix(sparse_vectors_rdd)

    # Get the number of rows and columns
    try:
        svd = row_matrix.computeSVD(num_concepts, computeU=True)
        U = svd.U
        S = svd.s
        V = svd.V

        # Assign IDs to concepts
        concept_ids = list(range(num_concepts))
        concept_strengths = list(zip(concept_ids, S))

        print("Assigned IDs and strengths for concepts:")
        for concept_id, strength in concept_strengths:
            print(f"ID: {concept_id}, strength: {strength}")

        return concept_ids, S, U, V
    except Exception as e:
        print(f"Error computing RowMatrix dimensions: {e}")
        return None

concept_ids, S, U, V = computeSVD(user_item_matrix)
print("Concepts and their singular values:", list(zip(concept_ids, S)))

Sample sparse vectors:
Sparse Vector 1: (467,[7,8,11,25,26,28,29,34,36,60,67,68,77,79,81,83,89,93,102,118,140,144,155,159,161,163,164,176,186,190,191,193,199,200,204,211,214,216,225,236,248,251,258,262,272,288,299,301,318,319,320,321,322,323,324,326,328,333,345,346,373,374,379,386,392,406,413,422,429,432,440,441,444,449,459,462],[3.5,3.5,2.5,2.0,3.5,2.5,2.0,3.0,3.0,2.0,1.5,3.0,3.5,2.0,3.0,2.0,2.5,2.5,2.5,2.5,2.0,1.0,2.5,1.5,2.5,3.0,2.5,4.0,2.5,2.0,3.0,2.0,3.0,2.5,2.5,1.5,3.0,2.5,3.5,0.5,2.5,3.5,3.5,2.0,2.5,2.5,3.0,1.5,2.0,3.0,2.0,3.0,1.5,3.0,3.0,3.0,3.0,3.5,2.0,3.0,2.5,2.0,2.5,2.0,1.5,3.0,3.0,4.0,4.5,3.5,1.0,3.5,3.5,2.5,3.5,3.5])
Sparse Vector 2: (467,[176],[1.0])
Sparse Vector 3: (467,[288,324,333,430,432],[4.5,3.0,4.5,5.0,5.0])
Sparse Vector 4: (467,[19,67,163,212,213,249,269,375,413],[3.0,3.0,4.0,4.0,3.0,2.5,3.0,1.5,4.5])
Sparse Vector 5: (467,[25,140,144,161,174,216,301,335,356,376,379,440],[5.0,5.0,5.0,3.0,5.0,4.0,5.0,5.0,5.0,5.0,5.0,5.0])
Sparse Vector 6: (467,[0,25,34,36,67,144,

In [ ]:
from pyspark.sql import Row
from pyspark.ml.linalg import VectorUDT, Vectors
from pyspark.sql.types import StructType, StructField, IntegerType

def map_user_concepts(U):
    u_with_ids = U.rows.zipWithIndex().map(
        lambda x: (x[1], int(max(range(len(x[0])), key=lambda i: abs(x[0][i]))), Vectors.dense(x[0]))
    )

    schema = StructType([
        StructField("user_id", IntegerType(), True),
        StructField("concept_id", IntegerType(), True),
        StructField("embedding", VectorUDT(), True)
    ])

    df_concept_user = spark.createDataFrame(u_with_ids, schema)

    return df_concept_user


def map_item_concepts(V):
    v_matrix_array = V.toArray()

    v_matrix_rows = [
        (i, Vectors.dense(row), max(range(len(row)), key=lambda k: abs(row[k])))
        for i, row in enumerate(v_matrix_array)
    ]

    df_concept_item = spark.createDataFrame(
        [Row(item_id=row[0], concept_id=row[2], embedding=row[1]) for row in v_matrix_rows]
    )
    return df_concept_item

df_concept_user = map_user_concepts(U)
df_concept_item = map_item_concepts(V)

print("User-to-concept:")
df_concept_user.show(truncate=False)

print("Item-to-concept:")
df_concept_item.show(truncate=False)


User-to-concept:
+-------+----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|concept_id|embedding                                                                                                                                                                                                                                          

In [ ]:
def compute_user_concept_portion(df_concept_user):
    # Count the number of users per concept
    user_concept_counts = df_concept_user.groupBy("concept_id").count()

    # Compute the portion of users per concept
    user_concept_portion = user_concept_counts.withColumn(
        "portion", F.col("count") / df_concept_user.count()
    )

    return user_concept_portion

def compute_item_concept_portion(df_concept_item):
    # Count the number of items per concept
    item_concept_counts = df_concept_item.groupBy("concept_id").count()

    # Compute the portion of items per concept
    item_concept_portion = item_concept_counts.withColumn(
        "portion", F.col("count") / df_concept_item.count()
    )

    return item_concept_portion

def compute_concept_portion(df_concept_user, df_concept_item):
    # Compute portions for users and items
    user_portion = compute_user_concept_portion(df_concept_user).withColumnRenamed("count", "user_count").withColumnRenamed("portion", "user_portion")
    item_portion = compute_item_concept_portion(df_concept_item).withColumnRenamed("count", "item_count").withColumnRenamed("portion", "item_portion")

    # Join on concept_id
    df_concept_portion = user_portion.join(item_portion, on="concept_id", how="outer")

    return df_concept_portion

# Compute concept portions
df_concept_portion = compute_concept_portion(df_concept_user, df_concept_item)

print("Concept Portion:")
df_concept_portion.show(truncate=False)

Concept Portion:
+----------+----------+--------------------+----------+--------------------+
|concept_id|user_count|user_portion        |item_count|item_portion        |
+----------+----------+--------------------+----------+--------------------+
|26        |5         |0.06756756756756757 |8         |0.017130620985010708|
|29        |4         |0.05405405405405406 |17        |0.03640256959314775 |
|19        |2         |0.02702702702702703 |6         |0.01284796573875803 |
|22        |2         |0.02702702702702703 |10        |0.021413276231263382|
|7         |1         |0.013513513513513514|22        |0.047109207708779445|
|31        |10        |0.13513513513513514 |11        |0.023554603854389723|
|25        |3         |0.04054054054054054 |13        |0.027837259100642397|
|6         |2         |0.02702702702702703 |20        |0.042826552462526764|
|9         |null      |null                |17        |0.03640256959314775 |
|27        |3         |0.04054054054054054 |13        |0.02

In [ ]:
import numpy as np
from pyspark.ml.linalg import VectorUDT, Vectors
from pyspark.sql.types import StructType, StructField

def compute_user_embeddings(U, S):
    # Scale U rows by singular values (S)
    scaled_u = U.rows.map(lambda row: np.array(row) * S)

    # Convert the resulting embeddings into DenseVectors and assign user_id
    user_embeddings = scaled_u.zipWithIndex().map(
        lambda x: (x[1], Vectors.dense(x[0]))  # Convert the array to a DenseVector
    )

    # Define schema for the DataFrame
    schema = StructType([
        StructField("user_id", IntegerType(), True),
        StructField("embedding", VectorUDT(), True)
    ])

    # Create DataFrame
    df_embedding_user = spark.createDataFrame(user_embeddings, schema)

    return df_embedding_user

# Compute user embeddings
df_embedding_user = compute_user_embeddings(U, S)

print("User Embeddings:")
df_embedding_user.show(truncate=False)

User Embeddings:
+-------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|embedding                                                                                                                                                                                                                                                                                                          

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, udf
from pyspark.sql.types import ArrayType, FloatType
from pyspark.ml.linalg import DenseVector

# Define UDF to convert DenseVector to a list
def dense_vector_to_list(v):
    if isinstance(v, DenseVector):
        return v.toArray().tolist()  # Convert DenseVector to list of floats
    return []

def save(df_concept_user, df_concept_item, df_embedding_user):
    # Register UDF
    vector_to_list_udf = udf(dense_vector_to_list, ArrayType(FloatType()))

    # Apply UDF to convert DenseVector to list for embeddings
    df_concept_user = df_concept_user.withColumn("embedding", vector_to_list_udf("embedding"))
    df_concept_item = df_concept_item.withColumn("embedding", vector_to_list_udf("embedding"))
    df_embedding_user = df_embedding_user.withColumn("embedding", vector_to_list_udf("embedding"))

    # Convert list of floats to comma-separated string for CSV storage
    df_concept_user = df_concept_user.withColumn("embedding", concat_ws(",", col("embedding")))
    df_concept_item = df_concept_item.withColumn("embedding", concat_ws(",", col("embedding")))
    df_embedding_user = df_embedding_user.withColumn("embedding", concat_ws(",", col("embedding")))

    # Write DataFrames to CSV files with header and coalesce to 1 file
    df_concept_user.coalesce(1).write.mode("overwrite").csv("concept_user.csv", header=True)
    df_concept_item.coalesce(1).write.mode("overwrite").csv("concept_item.csv", header=True)
    df_embedding_user.coalesce(1).write.mode("overwrite").csv("embedding_user.csv", header=True)

save(df_concept_user, df_concept_item, df_embedding_user)